# 1. 安装与环境配置


In [ ]:
# 安装 CCXT
%pip install ccxt pandas


## 创建交易所


In [ ]:
%cd /data/projects_ING/crypto_quant
# [ 创建交易所实例 ]
# import sys
# from pathlib import Path
# sys.path.append(str(Path(__file__).parent.parent))
import ccxt
import pandas as pd
from utils.config import config,logger
from utils.get_ccxt_kline import get_kline
# 加载交易所配置
exchange_name = "binance"
cfg=config.get('exchanges',{}).get(exchange_name)
# 创建交易所实例
exchange_class = getattr(ccxt, exchange_name)
exchange = exchange_class(cfg)
# 启用模拟交易
exchange.enable_demo_trading(True)


## 全仓模式


In [ ]:
## 全仓模式
# leverage=0 为全仓,gate
symbol="BTC/USDC:USDC"
try:
    exchange.set_margin_mode('cross', symbol)
except Exception as e:
    logger.error(f"设置全仓模式失败: {e}")


## 杠杆倍数


In [ ]:
symbol="BTC/USDC:USDC"
leverage=125
try:
    exchange.set_leverage(leverage, symbol)
except Exception as e:
    logger.error(f"设置杠杆失败: {e}")


## 双向持仓（高级）


In [ ]:
symbol="BTC/USDC:USDC"
try:
    exchange.set_position_mode(True, symbol)
except Exception as e:
    logger.error(f"设置仓位模式失败: {e}")


# 进场


## 回调进场

信号k出现,确认k出现,等回调 确认k的0.5 进场,


### 开多


In [ ]:
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '1m'
limit = 2
df = get_kline(exchange, symbol, timeframe, limit)

mid_price = (df.iloc[0]['close'] + df.iloc[0]['open'] ) / 2

buy_price = mid_price
amount=0.002
exchange.create_limit_buy_order(symbol, amount, buy_price, {
    'positionSide': 'LONG',
    'timeInForce': 'PO'
})
sl_price = df.iloc[0]['low'] 
exchange.create_limit_sell_order(symbol, amount, sl_price, {
    'positionSide': 'LONG',
    'stopLossPrice': sl_price,
    'timeInForce': 'PO'
})


### 开空


In [ ]:
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '15m'
limit = 2
df = get_kline(exchange, symbol, timeframe, limit)

mid_price = (df.iloc[0]['close'] + df.iloc[0]['open'] ) / 2

sell_price = mid_price
amount=0.002
exchange.create_limit_sell_order(symbol, amount, sell_price, {
    'positionSide': 'SHORT',
    'timeInForce': 'PO'
})

sl_price = df.iloc[0]['high']
exchange.create_limit_buy_order(symbol, amount, sl_price, {
    'positionSide': 'SHORT',
    'stopLossPrice': sl_price,
    'timeInForce': 'PO'
})


## 突破进场

close突破布林带时
多:high+20/low-20
空:low-20/high+20


### 开多


In [ ]:
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '1m'
limit = 2
df = get_kline(exchange, symbol, timeframe, limit)

buy_price,sl_price = df.iloc[0]['high']+20, df.iloc[0]['low']-20

amount=0.002
exchange.create_limit_buy_order(symbol, amount, buy_price, {
    'positionSide': 'LONG',
    'timeInForce': 'PO',
})
exchange.create_limit_sell_order(symbol, amount, sl_price, {
    'positionSide': 'LONG',
    'stopLossPrice': sl_price,
    'timeInForce': 'PO'
})


### 开空


In [ ]:
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '1m'
limit = 2
df = get_kline(exchange, symbol, timeframe, limit)

sell_price, sl_price =df.iloc[0]['low']-20 , df.iloc[0]['high']+20
amount=0.002

exchange.create_limit_sell_order(symbol, amount, sell_price, {
    'positionSide': 'SHORT',
    'timeInForce': 'PO',
})

# 2. 止损（用 GTC，不用 PO）
exchange.create_limit_buy_order(symbol, amount, sl_price, {
    'positionSide': 'SHORT',
    'stopLossPrice': sl_price,
    'timeInForce': 'PO'
})


## 反转进场

到达布林带边缘时,出现确认k后,反向开
LONG:close-20,stop:low-100
SHORT:close+20,stop:high+100


### 开多


In [ ]:
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '1m'
limit = 2
df = get_kline(exchange, symbol, timeframe, limit)

buy_price1 = df.iloc[0]['close']-20
amount=0.002
exchange.create_limit_buy_order(symbol, amount, buy_price1, {
    'positionSide': 'LONG',
    'timeInForce': 'PO'
})

sl_price = df.iloc[0]['low'] - 100
# 2. 止损（用 GTC，不用 PO）
exchange.create_limit_sell_order(symbol, amount, sl_price, {
    'positionSide': 'LONG',
    'stopLossPrice': sl_price,
    'timeInForce': 'PO'
})


### 开空


In [ ]:
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '1m'
limit = 2
df = get_kline(exchange, symbol, timeframe, limit)

sell_price1 = df.iloc[0]['close']+20  
amount=0.002
exchange.create_limit_sell_order(symbol, amount, sell_price1, {
    'positionSide': 'SHORT',
    'timeInForce': 'PO',
})
sl_price = df.iloc[0]['high']+100
# 2. 
exchange.create_limit_buy_order(symbol, amount*2, sl_price, {
    'positionSide': 'SHORT',
    'stopLossPrice': sl_price,
    'timeInForce': 'PO'
})


## 手动进场


### 多


In [ ]:
# 进场价
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '15m'
limit = 2
df = get_kline(exchange, symbol, timeframe, limit)
enter_price = df.iloc[-2]['close']

amount=0.002
exchange.create_limit_buy_order (symbol, amount, enter_price, {
    'positionSide': 'LONG',
    'timeInForce': 'PO'
})

sl_price = enter_price-1000
exchange.create_limit_sell_order(symbol, amount, sl_price, {
    'positionSide': 'LONG',
    'stopLossPrice': sl_price,
    'timeInForce': 'PO'
})
tp_price = enter_price+1000
exchange.create_limit_sell_order(symbol, amount, tp_price, {
    'positionSide': 'LONG',
    'takeProfitPrice': tp_price,
    'timeInForce': 'PO'
})


### 开空


In [ ]:
# 进场价
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '15m'
limit = 2
df = get_kline(exchange, symbol, timeframe, limit)

enter_price = df.iloc[0]['high']
print(f"进场价: {enter_price}")
amount=0.002
exchange.create_limit_sell_order (symbol, amount, enter_price, {
    'positionSide': 'SHORT',
    'timeInForce': 'PO'
})

sl_price = enter_price + 300
exchange.create_limit_buy_order(symbol, amount, sl_price, {
    'positionSide': 'SHORT',
    'stopLossPrice': sl_price,
    'timeInForce': 'PO'
})
tp_price = enter_price - 400
exchange.create_limit_buy_order(symbol, amount, tp_price, {
    'positionSide': 'SHORT',
    'takeProfitPrice': tp_price,
    'timeInForce': 'PO'
})


# 止损单（PostOnly）


### 移动止损多


In [ ]:
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '15m'
limit = 6
df = get_kline(exchange, symbol, timeframe, limit)

sl_price = df.iloc[0]['low']

# 获取多仓持仓
positions = exchange.fetch_positions([symbol])
for pos in positions:
    if pos['side'] == 'long' and float(pos.get('contracts', 0)) > 0:
        contracts = float(pos['contracts'])
                # 取消旧止损
        try:
            orders = exchange.fetch_open_orders(symbol, params={'stop': True})
            for order in orders:
                if order['side'] == 'sell':
                    exchange.cancel_order(order['id'], symbol, params={'stop': True})
        except:
            pass
        # 下止损单
        exchange.create_limit_sell_order(symbol, contracts, sl_price, {
            'positionSide': 'LONG',
            'stopLossPrice': sl_price
        })
        print(f"多仓止损已设置: {sl_price}")


### 移动止损空


In [ ]:
# 获取K线数据
symbol="BTC/USDC:USDC"
timeframe = '15m'
limit = 3
df = get_kline(exchange, symbol, timeframe, limit)

sl_price = df.iloc[0]['high'] 

# 获取空仓持仓
positions = exchange.fetch_positions([symbol])
for pos in positions:
    if pos['side'] == 'short' and float(pos.get('contracts', 0)) > 0:
        contracts = float(pos['contracts'])
        # 取消旧止损
        try:
            orders = exchange.fetch_open_orders(symbol, params={'stop': True})
            for order in orders:
                if order['side'] == 'buy':
                    exchange.cancel_order(order['id'], symbol, params={'stop': True})
        except:
            pass
        # 下止损单
        exchange.create_limit_buy_order(symbol, contracts, sl_price, {
            'positionSide': 'SHORT',
            'stopLossPrice': sl_price,
            'timeInForce': 'PO'
        })
        print(f"空仓止损已设置: {sl_price}")



### 限价买单


## 取消条件单


In [ ]:
# Binance 条件单需要特殊参数
try:
    # 获取条件单
    stop_orders = exchange.fetch_open_orders(symbol, params={'stop': True})
    
    # 取消条件单
    for order in stop_orders:
        exchange.cancel_order(order['id'], symbol, params={'stop': True})
        print(f"已取消条件单: {order['id']}")
except Exception as e:
    print(f"取消条件单失败: {e}")
    
# 取消全部挂单
# 取消指定交易对的所有挂单
exchange.cancel_all_orders(symbol)


## 取消全部挂单
